# Klasifikasi DemogPairs Menggunakan ViT (Emosi) & Logistic Regression

In [1]:
import numpy as np
import utils as u
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from imblearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm

joblib.parallel_backend('threading')

## Load Dataset

In [2]:
data = u.load_demogpairs()
pd.DataFrame(data)

,db_code,image_path,full_path,label,label_idx
0,CWF,able_wanamakok/002.jpg,dataset/demogpairs/images\able_wanamakok/002.jpg,Asian_Females,5
1,CWF,able_wanamakok/004.jpg,dataset/demogpairs/images\able_wanamakok/004.jpg,Asian_Females,5
2,CWF,able_wanamakok/007.jpg,dataset/demogpairs/images\able_wanamakok/007.jpg,Asian_Females,5
3,CWF,able_wanamakok/008.jpg,dataset/demogpairs/images\able_wanamakok/008.jpg,Asian_Females,5
4,CWF,able_wanamakok/012.jpg,dataset/demogpairs/images\able_wanamakok/012.jpg,Asian_Females,5
...,...,...,...,...,...
10795,CWF,zachary_quinto/177.jpg,dataset/demogpairs/images\zachary_quinto/177.jpg,White_Males,3
10796,CWF,zachary_quinto/214.jpg,dataset/demogpairs/images\zachary_quinto/214.jpg,White_Males,3
10797,CWF,zachary_quinto/217.jpg,dataset/demogpairs/images\zachary_quinto/217.jpg,White_Males,3
10798,CWF,zachary_quinto/218.jpg,dataset/demogpairs/images\zachary_quinto/218.jpg,White_Males,3


## Load Fitur

In [3]:
features = joblib.load('features/demogpairs_vit-emotion.pkl')
print('Jumlah fitur per gambar:', np.array(features[list(features.keys())[0]]).shape[0])

Jumlah fitur per gambar: 768


## Split Data

In [4]:
X = np.array([features[d['image_path']] for d in data])
y = np.array([d['label_idx'] for d in data])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print((len(X_train), len(X_test)))

(8640, 2160)


## Kombinasi Parameter

In [5]:
grid_params = [
    {
        'scaler': [None, MinMaxScaler()],
        'pca': [None, PCA(n_components=0.5), PCA(n_components=0.75)],
        
        'classifier': [LogisticRegression(random_state=42)],
        'classifier__C': [0.01, 0.1, 1, 10],
        'classifier__max_iter': [500, 1000],
        'classifier__solver': ['lbfgs', 'saga'],
    },
]

pipeline = Pipeline(steps=[
    ('scaler', None),
    ('pca', None),
    ('classifier', None)
])

skv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy', 
    'f1': 'f1_macro', 
    'precision': 'precision_macro', 
    'recall': 'recall_macro',
}

grid_models = {}
for params in grid_params:
    key = str(params['classifier'][0]).split('(')[0]
    grid_models[key] = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        cv=skv, refit='accuracy',
        scoring=scoring, n_jobs=int(joblib.cpu_count() * 0.6),
        verbose=1, error_score='raise',
        return_train_score=True
    )
    print(f'{key}: {len(ParameterGrid(params))} kombinasi')

LogisticRegression: 96 kombinasi


## Klasifikasi

In [6]:
evaluation_results, fold_results = u.evaluate_models(
    grid_models, 
    X_train, y_train,
    X_test, y_test,
    target_names=u.demogpairs_classes,
    model_prefix="models/clf_demogpairs_lr_vit-emotion_",
    results_path="results/demogpairs_lr_vit-emotion_"
)
sorted_results = pd.DataFrame(evaluation_results).sort_values(by="test_accuracy", ascending=False).to_dict("records")
u.html_br()
_dtable = u.display_table(sorted_results)

Evaluating: LogisticRegression


{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'saga', 'pca': None, 'scaler': 'MinMaxScaler'}


Accuracy  : 0.8847222222222222
Precision : 0.8850049960579972
Recall    : 0.8847222222222221
F1 Score  : 0.8846079825089609
               precision    recall  f1-score   support

Asian_Females     0.9101    0.9278    0.9188       360
  Asian_Males     0.8850    0.9194    0.9019       360
Black_Females     0.8678    0.8389    0.8531       360
  Black_Males     0.9218    0.9167    0.9192       360
White_Females     0.8974    0.8500    0.8730       360
  White_Males     0.8280    0.8556    0.8415       360

     accuracy                         0.8847      2160
    macro avg     0.8850    0.8847    0.8846      2160
 weighted avg     0.8850    0.8847    0.8846      2160



Class,OvR Accuracy,Precision,Recall,F1-Score,Support
Asian_Females,0.9726851851851852,0.9100817438692098,0.9277777777777778,0.9188445667125171,360
Asian_Males,0.9666666666666667,0.8850267379679144,0.9194444444444444,0.9019073569482288,360
Black_Females,0.9518518518518518,0.867816091954023,0.8388888888888889,0.8531073446327683,360
Black_Males,0.9731481481481481,0.9217877094972067,0.9166666666666666,0.9192200557103063,360
White_Females,0.9587962962962963,0.8973607038123167,0.85,0.8730385164051355,360
White_Males,0.9462962962962963,0.8279569892473119,0.8555555555555555,0.8415300546448087,360


Confusion matrix saved: images\cm_lr_vit-emotion_LogisticRegression.png



Confusion Matrix:
                         Asian_Females       Asian_Males     Black_Females       Black_Males     White_Females       White_Males
       Asian_Females               334                 1                11                 3                 9                 2
         Asian_Males                 1               331                 0                 7                 7                14
       Black_Females                 7                 1               302                17                 3                30
         Black_Males                 8                 7                14               330                 1                 0
       White_Females                14                19                 2                 1               306                18
         White_Males                 3                15                19                 0                15               308


model_name,model_file_path,best_parameters,test_accuracy,test_f1,test_precision,test_recall,parameter_combinations
LogisticRegression,models/clf_demogpairs_lr_vit-emotion_LogisticRegression.pkl,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'saga', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8847222222222222,0.8846079825089609,0.8850049960579972,0.8847222222222221,270


In [7]:
model, training_time = u.load_object('models/clf_demogpairs_lr_vit-emotion_LogisticRegression.pkl')
u.h(5, 'Waktu Pelatihan (Jobs)')
u.seconds_to_time(round(training_time))

{'input_seconds': 3233.0,
 'days': 0,
 'hours': 0,
 'minutes': 53,
 'seconds': 53.0,
 'text': '0 hari 0 jam 53 menit 53.0 detik'}

In [8]:
u.h(5, 'Waktu Pelatihan')
times = [fr['Train Time Mean'] * 5 for fr in fold_results]
u.seconds_to_time(round(np.sum(times) + model.refit_time_))

{'input_seconds': 36403.0,
 'days': 0,
 'hours': 10,
 'minutes': 6,
 'seconds': 43.0,
 'text': '0 hari 10 jam 6 menit 43.0 detik'}

In [9]:
_dtable = u.display_table(fold_results, n_items=[4, 4], column_widths=['5%', '45%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%', '5%'])

No,Params,Fold 1,Fold 2,Fold 3,Fold 4,Fold 5,Accuracy Mean,F1 Score Mean,Precision Mean,Recall Mean,Train Time Mean
1,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 2000, 'classifier__solver': 'saga', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8721,0.8686,0.8611,0.8756,0.8814,0.8718,0.8717,0.8723,0.8718,102.4163
2,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 1000, 'classifier__solver': 'saga', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8721,0.8686,0.8611,0.8756,0.8814,0.8718,0.8717,0.8723,0.8718,101.722
3,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 500, 'classifier__solver': 'saga', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8721,0.8686,0.8611,0.8756,0.8814,0.8718,0.8717,0.8723,0.8718,101.2053
4,"{'classifier': 'LogisticRegression', 'classifier__C': 1, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': None, 'scaler': 'MinMaxScaler'}",0.8727,0.8709,0.8617,0.8727,0.8808,0.8718,0.8717,0.8723,0.8718,50.8924
...,...,...,...,...,...,...,...,...,...,...,...
267,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'newton-cg', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7957,0.7969,0.7865,0.7905,0.805,0.7949,0.7946,0.7958,0.7949,5.0765
268,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 1000, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7957,0.7969,0.7853,0.7905,0.805,0.7947,0.7943,0.7956,0.7947,2.8051
269,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 2000, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7957,0.7969,0.7853,0.7905,0.805,0.7947,0.7943,0.7956,0.7947,3.0219
270,"{'classifier': 'LogisticRegression', 'classifier__C': 0.01, 'classifier__max_iter': 500, 'classifier__solver': 'lbfgs', 'pca': 'PCA', 'scaler': 'MinMaxScaler'}",0.7957,0.7969,0.7853,0.7905,0.805,0.7947,0.7943,0.7956,0.7947,3.2744
